# cdfi-stress-tester — CDFI Portfolio Stress Testing Engine
## Demo: Monte Carlo Stress Testing for CDFI Loan Portfolios

This notebook demonstrates how to use cdfi-stress-tester to:
- Generate realistic CDFI loan portfolios for testing
- Apply standard stress scenarios (2008 recession, COVID, rate spike)
- Run Monte Carlo simulations with correlated NOI/rate/property shocks
- Compute Value at Risk (VaR) at 95% and 99% confidence levels
- Analyze capital adequacy under stress
- Compare scenarios side-by-side

Background: Every CDFI runs portfolio stress tests for IC reporting,
board governance, and CDFI Fund compliance. The math is the same everywhere
but each CDFI builds it from scratch in Excel. This library standardizes it.


In [ ]:
import sys
sys.path.insert(0, '..')

from cdfistress import (
    generate_sample_portfolio,
    from_standard,
    create_recession_scenario, create_rate_shock_scenario, create_sector_specific_scenario,
    apply_shock_to_loan,
    MonteCarloEngine,
    build_correlation_matrix, default_correlations,
    value_at_risk, conditional_var, expected_loss, tail_loss,
    capital_adequacy, capital_adequacy_report, tier1_under_stress,
    generate_stress_report, scenario_comparison_table,
    STANDARD_SCENARIOS, SECTOR_DEFAULT_RATES,
)
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("cdfi-stress-tester loaded successfully")
print(f"\nStandard scenarios: {list(STANDARD_SCENARIOS.keys())}")
print(f"Sectors tracked:    {list(SECTOR_DEFAULT_RATES.keys())}")


## 1. Generate a Sample CDFI Portfolio

Realistic 50-loan CDFI portfolio across affordable housing, small business,
healthcare, education, and community facilities sectors.


In [ ]:
loans = generate_sample_portfolio(n=50, seed=42)

print(f"Generated {len(loans)} loans")
total_balance = sum(l.outstanding_balance for l in loans)
total_committed = sum(l.commitment_amount for l in loans)
print(f"Total Outstanding:  ${total_balance/1e6:.2f}MM")
print(f"Total Committed:    ${total_committed/1e6:.2f}MM")

# Show first few loans
print("\nSample loans:")
print("=" * 75)
for loan in loans[:5]:
    print(f"  {loan.id:<10} {loan.sector:<22} ${loan.outstanding_balance/1e6:>5.2f}MM  DSCR: {loan.dscr:.2f}x  LTV: {loan.ltv*100:.0f}%")


## 2. Standard Stress Scenarios

The library ships with pre-calibrated scenarios based on real historical events.


In [ ]:
print("Standard Stress Scenarios:")
print("=" * 80)
for key in STANDARD_SCENARIOS:
    s = from_standard(key)
    print(f"\n  {s.name} [{s.severity}]")
    print(f"    NOI shock:           {s.noi_shock*100:>6.1f}%")
    print(f"    Rate shock:          {s.rate_shock*100:>6.1f}%")
    print(f"    Property value:      {s.property_value_shock*100:>6.1f}%")
    print(f"    Default multiplier:  {s.default_rate_multiplier:>6.2f}x")


## 3. Build the Monte Carlo Engine

The engine generates correlated shocks to NOI, interest rates, and property
values using multivariate normal distributions.


In [ ]:
engine = MonteCarloEngine(loans=loans, available_capital=5_000_000)

print(f"Engine initialized")
print(f"Loans:              {len(engine.loans)}")
print(f"Available capital:  ${engine.available_capital:,.0f}")
print(f"\nDefault correlations (NOI, Rate, PropertyValue):")
print(default_correlations())


## 4. Run a 2008-Style Recession Scenario

1,000 Monte Carlo iterations applying severe recession shocks to the portfolio.


In [ ]:
scenario = from_standard("2008_recession")
result = engine.run_simulation(scenario, n_iterations=1000, seed=42)

print(f"Scenario: {result.scenario.name}")
print(f"Severity: {result.scenario.severity.upper()}")
print("=" * 50)
print(f"Expected Loss:        ${result.expected_loss:,.0f}")
print(f"VaR (95%):            ${result.var_95:,.0f}")
print(f"VaR (99%):            ${result.var_99:,.0f}")
print(f"Capital Adequacy:     {result.capital_adequacy_ratio:.2f}x")
print(f"Buffer Breaches:      {result.num_breaches} of 1000 simulations")


## 5. Generate Full Stress Report

In [ ]:
report = generate_stress_report(result)
print(report)


## 6. Compare Multiple Scenarios

Run the portfolio through each standard scenario to compare outcomes.


In [ ]:
scenarios = [from_standard(k) for k in STANDARD_SCENARIOS.keys()]
results = [engine.run_simulation(s, n_iterations=500, seed=0) for s in scenarios]

comparison = scenario_comparison_table(results)
print("Scenario Comparison Table:")
print("=" * 90)
print(comparison.to_string(index=False))


## 7. Custom Scenarios

Build your own scenarios — e.g. CRE-specific stress or rate spike only.


In [ ]:
# Custom rate spike scenario
custom = create_rate_shock_scenario(
    name="500 bps Rate Spike",
    rate_shock=0.05,
)

result_custom = engine.run_simulation(custom, n_iterations=500, seed=0)
print(f"Custom Scenario: {custom.name}")
print(f"Expected Loss:    ${result_custom.expected_loss:,.0f}")
print(f"VaR (95%):        ${result_custom.var_95:,.0f}")


## 8. Sector-Specific Stress

Test what happens if affordable housing alone goes into severe distress.


In [ ]:
sector_scenario = create_sector_specific_scenario(
    sector="affordable_housing",
    name="Affordable Housing Crash",
    sector_default_multiplier=3.0,
)

result_sector = engine.run_simulation(sector_scenario, n_iterations=500, seed=0)
print(f"Scenario: {sector_scenario.name}")
print(f"Expected Loss:    ${result_sector.expected_loss:,.0f}")
print(f"VaR (95%):        ${result_sector.var_95:,.0f}")


## 9. Capital Adequacy Deep Dive

Analyze whether the CDFI has enough capital to absorb stress-case losses.


In [ ]:
# Generate a loss distribution from the simulation
losses = result.expected_loss * np.random.default_rng(0).lognormal(0, 0.5, 1000)

cap_report = capital_adequacy_report(
    available_capital=5_000_000,
    loss_distribution=losses,
)
print("Capital Adequacy Report:")
print("=" * 50)
for k, v in cap_report.items():
    if isinstance(v, float):
        print(f"  {k:<30} ${v:,.0f}" if v > 100 else f"  {k:<30} {v:.4f}")
    else:
        print(f"  {k:<30} {v}")


## 10. Risk Metric Deep Dive

VaR, CVaR, and tail loss for any loss distribution.


In [ ]:
var_95 = value_at_risk(losses, confidence=0.95)
var_99 = value_at_risk(losses, confidence=0.99)
cvar_95 = conditional_var(losses, confidence=0.95)
exp_loss = expected_loss(losses)
tail = tail_loss(losses, percentile=99)

print("Risk Metrics from Loss Distribution:")
print("=" * 50)
print(f"  Mean Expected Loss:      ${exp_loss:,.0f}")
print(f"  VaR at 95%:              ${var_95:,.0f}")
print(f"  VaR at 99%:              ${var_99:,.0f}")
print(f"  CVaR at 95% (Expected Shortfall): ${cvar_95:,.0f}")
print(f"  Tail Loss (99th pctile): ${tail:,.0f}")


## Summary

This notebook demonstrated the complete CDFI stress testing workflow:

1. **Generate realistic CDFI portfolios** — 50 loans across sectors
2. **Standard stress scenarios** — 2008 recession, COVID, rate spike, regional CRE crash
3. **Monte Carlo simulation** — 1,000 iterations with correlated shocks
4. **Stress reports** — formatted Markdown summaries
5. **Scenario comparison** — side-by-side results across scenarios
6. **Custom scenarios** — build your own stress tests
7. **Sector-specific stress** — drill into single sector exposures
8. **Capital adequacy** — does the CDFI have enough buffer?
9. **Risk metrics** — VaR, CVaR, tail loss deep dive

**Key use case:** CDFIs use this for IC reporting, board presentations,
CDFI Fund examinations, and capital planning. Combined with `impact-ledger`
for portfolio tracking and `dscr-tools` for loan-level analytics, this
library completes a full open source CDFI risk management toolkit.

**GitHub:** https://github.com/Jaypatel1511/cdfi-stress-tester
**PyPI:** https://pypi.org/project/cdfi-stress-tester
